<div align="center">

# Week 08: Networks and Communication: Modbus, MQTT, IEC 61850 and Control over Networks

**Cyber-Physical Systems (BIL-435), hands-on Colab notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Week overview](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/weeks/week-08) · [Lecture page](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-08/lecture.html) · [Interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-08/lab.html) · [PDF notes](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-08/Week08_Lecture_Notes.pdf)

</div>

## How to use this Colab notebook

This Colab notebook is the hands-on part of the week. The concepts are explained on the [lecture page](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-08/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows. They have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from an orange Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures. When the Colab notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback. It also stores the constants of the frequency model of the campus microgrid and the three devices on the Modbus bus of the battery container. The second cell keeps the expected values of the exercises in the table `ANSWERS`, with its code hidden, so that no answer stands next to its exercise.

In [ ]:
# Environment: Everything used by this course is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import io
import math
import struct
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (7, 3.6),
                     "axes.grid": True, "grid.alpha": 0.3})

def check(name, value, expected=None, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback.
    Without an expected value in the call, check takes the value and its tolerance
    from ANSWERS, the table of the next cell."""
    if expected is None:
        answer, tol = ANSWERS[name]
        expected = answer()
    if value is None:
        print(f"{name}: not answered yet. Replace None and run the cell again.")
        return False
    try:
        scale = max(1.0, abs(float(expected)))
        ok = abs(float(value) - float(expected)) <= tol * scale
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else
                         "not yet. Compare your steps with the lecture."))
    return ok

# The campus microgrid in island mode (Week 2): inertia M, damping D, droop gain K
M_KWS_HZ, D_KW_HZ, K_KW_HZ = 80.0, 12.0, 125.0
# The Modbus bus of the battery container: device, address, registers per poll
BUS = [("container meter", 1, 10), ("battery inverter", 2, 20), ("battery BMS", 3, 40)]
print("Setup complete. Running in Colab:", IN_COLAB, "| M, D, K =", M_KWS_HZ, D_KW_HZ,
      K_KW_HZ, "| devices on the bus:", [b[0] for b in BUS])

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
ANSWERS = {
    "Exercise 8.1": (lambda: 0x2A, 1e-9),
    "Exercise 8.2": (lambda: (0b10110110 >> 4) & 0x0F, 1e-9),
    "Exercise 8.3": (lambda: struct.unpack('>H', bytes([0x01, 0xF4]))[0] / 10, 1e-9),
    "Exercise A": (lambda: (8 + 5 + 2 * 20) * 11 / 9600 * 1000 + 2 * 3.5 * 11 / 9600 * 1000 + 5, 0.05),
    "Exercise B": (lambda: len([t for t in TOPICS if matches('campus/+/meter/power', t)]), 1e-9),
    "Exercise C": (lambda: 1 / (500 - 450) * 1000, 1e-6),
    "Exercise D": (lambda: math.acos(-0.1 / 1.0) / math.sqrt(1.0 ** 2 - 0.1 ** 2), 1e-4),
    "Exercise E": (lambda: float(STAGE8["delay margin kept, campus server (%)"]), 1e-6),
}

## Python step 8: Bytes, bits and binary messages

Devices on a fieldbus exchange bytes, not numbers or text. This step builds such messages in Python. It covers bytes, their hexadecimal notation and the bitwise operators that read single bits. It also packs numbers into 16-bit registers and computes a checksum over a message with a loop &#91;[1](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-08/lecture.html#ref-1)&#93;.

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "BUS" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-08/"
               "NB08_networks_communication.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

### Bytes and hexadecimal

In [ ]:
# Python practice: a small example for learning the language
msg = bytes([1, 3, 0, 0, 0, 10])      # six bytes, each from 0 to 255
print(msg)
print("as hexadecimal:", msg.hex(" ").upper())
print("first byte:", msg[0], "| last byte:", msg[-1], "| length:", len(msg))
print("250 in hexadecimal:", hex(250), "| 0xFA as a number:", 0xFA)

A `bytes` object is a sequence of small whole numbers from 0 to 255, one per byte. Hexadecimal writes each byte as two digits from 0 to F, so `0A` is 10 and `FA` is 250. The method `.hex(" ")` prints the bytes in this notation with spaces, the way a protocol analyser shows a frame.

### Bits and bitwise operators

In [ ]:
# Python practice: a small example for learning the language
status = 0b00101101                   # a status byte written in binary
print("status:", status, "=", format(status, "08b"))
alarm = (status >> 3) & 1              # bit 3: shift right by 3, keep the last bit
print("bit 3 (alarm):", alarm)
print("high nibble:", status >> 4, "| low nibble:", status & 0x0F)
print("set bit 1:", format(status | 0b10, "08b"), "| flip all:", format(status ^ 0xFF, "08b"))

The operator `>>` shifts the bits to the right, and `&` keeps only the bits that are 1 in both numbers. Together they read one bit or a group of bits, for example the alarm bit of a status word. The operator `|` sets bits and `^` flips them. `format(x, "08b")` writes a number as eight binary digits.

### Packing numbers into registers

In [ ]:
# Python practice: a small example for learning the language
power_w = 48250                        # a power of 48.25 kW in watts
two = struct.pack(">I", power_w)        # 32-bit unsigned, big-endian: four bytes
print("four bytes:", two.hex(" ").upper())
hi, lo = struct.unpack(">HH", two)      # the same bytes as two 16-bit registers
print("registers:", hi, lo, "| back together:", (hi << 16) | lo)
f = struct.pack(">f", 49.98)            # a float in four bytes (IEEE 754)
print("49.98 Hz as a float:", f.hex(" ").upper(), "->", round(struct.unpack(">f", f)[0], 4))

A Modbus register holds 16 bits. A larger number, such as a power in watts or a frequency as a floating-point value, occupies two registers. The module `struct` turns numbers into bytes and back. In its format strings, `>` means big-endian, the most significant byte first. The letters `H` and `I` stand for unsigned integers of 16 and 32 bits, and `f` for a 32-bit float.

### A checksum loop

In [ ]:
# Python practice: a small example for learning the language
def checksum8(data):
    """The sum of all bytes, kept to one byte: a very simple check."""
    total = 0
    for b in data:
        total = (total + b) & 0xFF
    return total

frame = bytes([1, 3, 0, 0, 0, 10])
print("checksum:", checksum8(frame))
damaged = bytes([1, 3, 0, 0, 0, 11])     # one bit changed on the line
print("after one flipped bit:", checksum8(damaged))

A checksum lets the receiver test whether a message arrived unchanged: The sender appends it, and the receiver computes it again. `& 0xFF` keeps the sum within one byte. Section 1 replaces this simple sum with the cyclic redundancy check of Modbus, which finds many more kinds of errors.

**Quick check.** Why does a frequency of 49.98 Hz need two Modbus registers when it is sent as a float?

<details><summary>Answer</summary>

A 32-bit float has four bytes, and a register holds two bytes, so the value spans two registers.

</details>

### Exercises for Python step 8

**Exercise 8.1.** A frame starts with the byte `0x2A`, the address of the device. Store this address as a number in `device_id`.

In [ ]:
device_id = None   # your answer

check("Exercise 8.1", device_id)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
device_id = bytes([0x2A, 0x03])[0]

check("Exercise 8.1", device_id)

**Exercise 8.2.** Store the high nibble, the upper four bits, of the byte `0b10110110` in `nibble`. Use `>>` and `&`.

In [ ]:
nibble = None   # your answer

check("Exercise 8.2", nibble)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
nibble = (0b10110110 >> 4) & 0x0F

check("Exercise 8.2", nibble)

**Exercise 8.3.** A meter sends the voltage as one register in tenths of a volt. The register arrives as the bytes `01 F4`. Unpack it with `struct` and store the voltage in volts in `volts`.

In [ ]:
volts = None   # your answer

check("Exercise 8.3", volts)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
volts = struct.unpack(">H", bytes([0x01, 0xF4]))[0] / 10

check("Exercise 8.3", volts)

## 1. Modbus RTU: Frames, checksums and poll cycles &#91;[4](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-08/lecture.html#ref-4)&#93;

*Lecture: [Modbus: Polling the field devices](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-08/lecture.html#modbus)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "checksum8" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-08/"
               "NB08_networks_communication.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A Modbus master reads its devices one after another, and every byte takes time on the serial line. Python builds the request frames with their checksum and computes how long one poll cycle of the battery container takes.

**From the formulas to the code.** The character time $t_c = 11 / r$ is `11 / baud`, and the silent interval $t_{3.5}$ is `t35(baud)`. The time of one transaction, $T_{dev} = (8 + 5 + 2N)\,t_c + 2\,t_{3.5} + t_{resp}$, is `poll_time(n_reg, baud)`, and the cycle $T_{cycle} = \sum T_{dev}$ is `cycle_time(BUS, baud)`.

In [ ]:
def crc16_modbus(data):
    """CRC-16 of Modbus RTU: initial value 0xFFFF, reflected polynomial 0xA001."""
    crc = 0xFFFF
    for b in data:
        crc ^= b
        for _ in range(8):
            crc = (crc >> 1) ^ 0xA001 if crc & 1 else crc >> 1
    return crc

def read_request(address, start, count):
    """Function 03, Read Holding Registers, with the CRC appended low byte first."""
    pdu = bytes([address, 0x03]) + struct.pack(">HH", start, count)
    crc = crc16_modbus(pdu)
    return pdu + bytes([crc & 0xFF, crc >> 8])

req = read_request(1, 0, 10)
print("request:", req.hex(" ").upper(), "| bytes:", len(req))
print("CRC of the received frame:", hex(crc16_modbus(req)), "(0x0 means intact)")
bad = bytearray(req)
bad[4] ^= 0x04                                  # one bit flipped on the line
print("CRC after one flipped bit:", hex(crc16_modbus(bytes(bad))))

**Python note.** `crc ^= b` mixes the next byte into the register, and the inner loop shifts it bit by bit, applying the polynomial whenever a 1 falls out. Computed over a frame that includes its own CRC, the result is 0, which is how the receiver checks a frame.

In [ ]:
def t35(baud):
    """Silent interval between frames in ms: 3.5 characters, fixed at 1.75 ms above 19200 baud."""
    return 1.75 if baud > 19200 else 3.5 * 11 / baud * 1000

def poll_time(n_reg, baud, t_resp=5.0):
    """One transaction in ms: request (8 bytes), response (5 + 2N bytes), two silences, the reply delay."""
    t_char = 11 / baud * 1000
    return (8 + 5 + 2 * n_reg) * t_char + 2 * t35(baud) + t_resp

def cycle_time(bus, baud, t_resp=5.0):
    return sum(poll_time(n, baud, t_resp) for _, _, n in bus)

print(f"worked example, meter at 9600 baud: {poll_time(10, 9600):.2f} ms")
for baud in [9600, 19200, 38400, 115200]:
    parts = [round(poll_time(n, baud), 2) for _, _, n in BUS]
    print(f"{baud:>6} baud: {parts} -> cycle {cycle_time(BUS, baud):.1f} ms")

**Python note.** Above 19200 baud the specification fixes the silent interval at 1.75 ms, so the function returns that value instead of 3.5 character times. The default `t_resp=5.0` is the time a device needs before it answers.

**Exercise A.** Store the time in ms of one transaction with the battery inverter, 20 registers at 9600 baud and a reply delay of 5 ms, in `t_inv`.

In [ ]:
t_inv = None   # your answer

check("Exercise A", t_inv)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
t_inv = poll_time(20, 9600)

check("Exercise A", t_inv)

Part A of the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-08/lab.html) ends with the button *Copy the results for Colab*. Paste the copied text between the triple quotes of `MY_BENCH`. Left empty, the variable makes the code use the rounds of a simulated student.

In [ ]:
# Paste the results of lab Part A between the triple quotes.
MY_BENCH = """
"""

if MY_BENCH.strip():
    bench = pd.read_csv(io.StringIO(MY_BENCH.strip()))
    source = "your own rounds on the Modbus bench"
else:
    bench = pd.DataFrame({"round": [1, 2, 3, 4], "baud": [9600, 9600, 9600, 19200],
                          "devices": [1, 3, 3, 3], "predicted_ms": [50.8, 230.0, 244.0, 130.0],
                          "measured_ms": [51.1, 244.5, 244.4, 129.9]})
    source = "a simulated student"
models = []
for _, r in bench.iterrows():
    models.append(cycle_time(BUS[:int(r["devices"])], int(r["baud"])))
bench["model_ms"] = np.round(models, 1)
bench["error_of_prediction_%"] = np.round(100 * (bench["predicted_ms"] - bench["model_ms"])
                                          / bench["model_ms"], 1)
print("Data:", source)
print(bench.to_string(index=False))

**Python note.** `iterrows()` walks through the rows of a table, and `BUS[:k]` keeps the first `k` devices, so each round is compared with the model of the devices it polled.

In [ ]:
# Figure: one poll cycle on the line and the cycle time against the baud rate
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
t, colours = 0.0, ["#1E6FA8", "#C98510", "#2E8B57"]
for (name, addr, n), c in zip(BUS, colours):
    tc = 11 / 9600 * 1000
    for label, dur, col in [("request", 8 * tc, c), ("silence", t35(9600), "#D3DCE3"),
                            ("reply delay", 5.0, "#EEF2F5"), ("response", (5 + 2 * n) * tc, c),
                            ("silence", t35(9600), "#D3DCE3")]:
        ax[0].barh(0, dur, left=t, color=col, edgecolor="white", alpha=0.95 if col == c else 1)
        t += dur
    ax[0].text(t - 60, 0.45, f"{name}", fontsize=8)
ax[0].set_yticks([])
ax[0].set_xlabel("time (ms) at 9600 baud")
ax[0].set_title(f"One poll cycle: {cycle_time(BUS, 9600):.1f} ms", fontsize=10)
bauds = np.array([4800, 9600, 19200, 38400, 57600, 115200])
ax[1].plot(bauds, [cycle_time(BUS, b) for b in bauds], "o-", color="#1E6FA8")
ax[1].axhline(150, ls=":", color="#B53F2E", label="target 150 ms")
ax[1].set_xscale("log")
ax[1].set_xlabel("baud rate")
ax[1].set_ylabel("poll cycle (ms)")
ax[1].legend()
plt.tight_layout()
plt.show()

**Observe and reflect.** Which part of a transaction shrinks when the baud rate doubles, and which part does not? Why does the cycle hardly improve beyond 57600 baud?

## 2. MQTT: Topics, wildcards and traffic &#91;[6](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-08/lecture.html#ref-6)&#93;

*Lecture: [MQTT: Publish and subscribe](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-08/lecture.html#mqtt)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "cycle_time" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-08/"
               "NB08_networks_communication.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A broker forwards each message to every client whose topic filter matches its topic. Python matches filters with wildcards, checks the filters of Part B and counts the packets that the broker handles at each quality of service.

**From the formulas to the code.** The matching rules of `+` and `#` are written in `matches(filter, topic)`. The packets per message, $k = 1, 2, 4$ for QoS 0, 1 and 2, are `PACKETS[qos]`, and the packets at the broker, $n\,(k_{pub} + S\,k_{sub})$, are `broker_packets(n, s, qos)`.

In [ ]:
TOPICS = ["campus/b1/meter/power", "campus/b1/meter/energy", "campus/b2/meter/power",
          "campus/b2/meter/energy", "campus/b3/meter/power", "campus/b3/hvac/temp",
          "campus/pv/inverter/power", "campus/wind/turbine/power", "campus/bat/bms/soc",
          "campus/bat/inverter/power", "campus/pcc/meter/frequency", "campus/pcc/meter/power"]

def matches(filt, topic):
    """MQTT topic matching: '+' matches one level, '#' matches the rest (only at the end)."""
    f, t = filt.split("/"), topic.split("/")
    for i, part in enumerate(f):
        if part == "#":
            return True
        if i >= len(t) or (part != "+" and part != t[i]):
            return False
    return len(f) == len(t)

for filt in ["campus/#", "campus/+/meter/power", "campus/bat/#", "campus/+/+/power", "campus/b2/+"]:
    hits = [t for t in TOPICS if matches(filt, t)]
    print(f"{filt:<22} -> {len(hits):>2} topics: {', '.join(h.split('/', 1)[1] for h in hits)}")

**Python note.** `split("/")` cuts a topic into its levels. The loop compares level by level: `+` accepts any one level, and `#` accepts everything that follows. Without `#`, filter and topic must have the same number of levels.

**Exercise B.** Store in `n_meter_power` how many of the `TOPICS` the filter `campus/+/meter/power` matches.

In [ ]:
n_meter_power = None   # your answer

check("Exercise B", n_meter_power)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n_meter_power = sum(matches("campus/+/meter/power", t) for t in TOPICS)

check("Exercise B", n_meter_power)

Part B of the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-08/lab.html) records the filters that you typed for each mission. Paste the copied text into `MY_TOPICS`, and the code checks every filter against the rules above.

In [ ]:
# Paste the results of lab Part B between the triple quotes.
MY_TOPICS = """
"""
MISSIONS = {1: ["campus/#"], 2: ["campus/b2/#"], 3: ["campus/+/meter/power"],
            4: ["campus/bat/#"], 5: ["campus/+/+/power"]}

def hits_of(filters):
    return {t for t in TOPICS if any(matches(f.strip(), t) for f in filters if f.strip())}

if MY_TOPICS.strip():
    tries = pd.read_csv(io.StringIO(MY_TOPICS.strip()))
    tries = tries[tries["mission"].isin(list(MISSIONS))]     # mission 6 asks for a number
    source = "your own filters"
else:
    tries = pd.DataFrame({"mission": [1, 2, 3, 3, 4, 5],
                          "filters": ["campus/#", "campus/b2/#", "campus/+/power",
                                      "campus/+/meter/power", "campus/bat/+/+", "campus/+/+/power"]})
    source = "a simulated student"
rows = []
for _, r in tries.iterrows():
    got, want = hits_of(str(r["filters"]).split(";")), hits_of(MISSIONS[int(r["mission"])])
    rows.append([int(r["mission"]), r["filters"], len(got), len(want), len(got - want),
                 len(want - got), got == want])
check_table = pd.DataFrame(rows, columns=["mission", "filters", "received", "wanted",
                                          "extra", "missing", "correct"])
print("Data:", source)
print(check_table.to_string(index=False))

**Python note.** A set keeps each topic once, and `got - want` holds the topics that a filter receives without being asked for. Several filters of one subscriber are written in one cell, separated by semicolons.

In [ ]:
PACKETS = {0: 1, 1: 2, 2: 4}      # PUBLISH; PUBLISH + PUBACK; PUBLISH, PUBREC, PUBREL, PUBCOMP

def broker_packets(n_msgs, subscribers, qos):
    """Packets that pass the broker: one exchange from the publisher, one to each subscriber."""
    return n_msgs * (PACKETS[qos] + subscribers * PACKETS[qos])

n_hour = 12 * 3600 // 25        # 12 meters, one message every 25 s
for q in (0, 1, 2):
    print(f"QoS {q}: {broker_packets(n_hour, 3, q):,} packets per hour for {n_hour:,} messages")

def on_change(series, deadband, heartbeat):
    """Indices of the samples that are published: a change beyond the deadband or a heartbeat."""
    sent, last_value, last_time = [], None, -10 ** 9
    for k, v in enumerate(series):
        if last_value is None or abs(v - last_value) > deadband or k - last_time >= heartbeat:
            sent.append(k)
            last_value, last_time = v, k
    return sent

rng = np.random.default_rng(RANDOM_STATE)
steps = 3600 // 25
loads = 150 + np.cumsum(rng.normal(0, 1.5, size=(12, steps)), axis=1)   # kW, a slow drift
loads[:, steps // 2:] += rng.choice([0, 22], size=(12, 1))             # a charger switches on
sent = sum(len(on_change(row, deadband=5.0, heartbeat=12)) for row in loads)
print(f"periodic: {12 * steps} messages per hour | on change (5 kW, heartbeat 5 min): {sent}")

**Python note.** `np.cumsum` adds up random steps, so each row drifts like a slowly changing load. A message is sent when the value has moved by more than the deadband since the last message, or when 12 samples, five minutes, have passed without one.

In [ ]:
# Figure: messages per hour against the deadband
bands = [0.0, 1, 2, 3, 5, 8, 12, 20]
counts = [sum(len(on_change(row, b, 12)) for row in loads) for b in bands]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(bands, counts, "o-", color="#1E6FA8")
ax[0].axhline(12 * steps, ls=":", color="#585E6E", label="periodic, every 25 s")
ax[0].axhline(12 * 12, ls="--", color="#C98510", label="heartbeats only")
ax[0].set_xlabel("deadband (kW)")
ax[0].set_ylabel("messages per hour")
ax[0].legend(fontsize=8)
row, idx = loads[0], on_change(loads[0], 5.0, 12)
ax[1].plot(np.arange(steps) * 25 / 60, row, color="#585E6E", lw=1, label="load of meter 1")
ax[1].plot(np.array(idx) * 25 / 60, row[idx], "o", color="#1E6FA8", ms=4, label="published")
ax[1].set_xlabel("time (min)")
ax[1].set_ylabel("power (kW)")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Observe and reflect.** Which deadband would you choose for the meters of the campus, and what would a dashboard lose with a deadband of 20 kW? Why does the heartbeat matter for a subscriber that has just connected?

## 3. GOOSE repetitions and queues at a gateway &#91;[10](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-08/lecture.html#ref-10)&#93;

*Lecture: [Latency, queues and repetition](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-08/lecture.html#latency)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "broker_packets" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-08/"
               "NB08_networks_communication.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A trip message that is lost must be repeated in time, and a message that waits in a queue arrives late. Python computes the chance that every repetition of a trip is lost, and simulates a gateway as a queue to check the formulas of the lecture.

**From the formulas to the code.** The probability of a missed trip, $p^k$ for $k$ independent messages within the deadline, is `p_miss(p, k)`. For the queue, $\rho = \lambda / \mu$, $W = 1 / (\mu - \lambda)$ and $L = \lambda W$ are computed in `mm1(lam, mu)` and compared with `simulate_queue(lam, mu)`.

In [ ]:
def goose_times(n=8, first=2.0):
    """Send times in ms after an event: 0, then first, 2 first, 4 first, ... (doubling)."""
    return [0.0] + [first * 2 ** k for k in range(n - 1)]

def in_time(times, transfer, deadline):
    """Repetitions that arrive before the deadline."""
    return [t for t in times if t + transfer <= deadline]

def p_miss(p, k):
    return p ** k

times = goose_times()
ok = in_time(times, transfer=1.5, deadline=10.0)
print("send times (ms):", times)
print("arrive within 10 ms:", ok, "->", len(ok), "messages")
for p in [0.01, 0.02, 0.05]:
    print(f"loss {p:.2f} per message: P(all {len(ok)} lost) = {p_miss(p, len(ok)):.2e}")

**Python note.** `2 ** k` doubles the interval with every repetition. The probability $p^k$ assumes that the losses are independent; a cut cable loses all repetitions at once, which is why protection networks are often duplicated.

In [ ]:
def mm1(lam, mu):
    rho = lam / mu
    W = 1 / (mu - lam)
    return rho, W, lam * W

def simulate_queue(lam, mu, n=50000, seed=RANDOM_STATE):
    """First come, first served: exponential gaps between arrivals and exponential service."""
    rng = np.random.default_rng(seed)
    arrive = np.cumsum(rng.exponential(1 / lam, n))
    service = rng.exponential(1 / mu, n)
    done = np.empty(n)
    free = 0.0
    for i in range(n):
        start = max(arrive[i], free)
        free = done[i] = start + service[i]
    return done - arrive

for lam in [200, 400, 450, 480]:
    rho, W, L = mm1(lam, 500)
    d = simulate_queue(lam, 500)
    print(f"lambda {lam}/s: rho {rho:.2f}, W {1000 * W:5.1f} ms (simulated {1000 * d.mean():5.1f}), "
          f"L {L:5.2f}, 99th percentile {1000 * np.percentile(d, 99):6.1f} ms")

**Python note.** Each message starts its service when it arrives or when the previous message is done, whichever is later. `done - arrive` is the time each message spends at the gateway, waiting plus service.

**Exercise C.** A gateway serves 500 messages per second, and 450 arrive per second. Store the mean time in ms that a message spends at the gateway in `w_ms`.

In [ ]:
w_ms = None   # your answer

check("Exercise C", w_ms)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
w_ms = 1000 * mm1(450, 500)[1]

check("Exercise C", w_ms)

In [ ]:
# Figure: delay against load, formula and simulation
lams = np.array([50, 100, 200, 300, 350, 400, 425, 450, 475])
sim = [simulate_queue(l, 500, n=20000) for l in lams]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
rr = np.linspace(0.05, 0.96, 100)
ax[0].plot(rr, 1000 / (500 * (1 - rr)), color="#1E6FA8", label="W = 1 / (mu - lambda)")
ax[0].plot(lams / 500, [1000 * d.mean() for d in sim], "o", color="#C98510", label="simulated mean")
ax[0].plot(lams / 500, [1000 * np.percentile(d, 99) for d in sim], "s", color="#B53F2E", ms=4,
           label="simulated 99th percentile")
ax[0].set_xlabel("utilisation rho")
ax[0].set_ylabel("time at the gateway (ms)")
ax[0].set_yscale("log")
ax[0].legend(fontsize=8)
ax[1].hist(1000 * sim[7], bins=60, color="#1E6FA8", alpha=0.8)
ax[1].set_xlabel("time at the gateway (ms), rho = 0.9")
ax[1].set_ylabel("messages")
plt.tight_layout()
plt.show()

**Observe and reflect.** How much later does the slowest message of a hundred arrive than the average one at a utilisation of 0.9? What would you tell an engineer who sizes a gateway for its average load?

## 4. Frequency control over the network: Your delays &#91;[12](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-08/lecture.html#ref-12)&#93;

*Lecture: [Control over a network](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-08/lecture.html#control-over-a-network)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "simulate_queue" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-08/"
               "NB08_networks_communication.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A controller that receives old measurements acts on an old state of the plant. Python simulates the frequency of the campus microgrid with a delayed controller, finds the delay at which it becomes unstable and compares it with the formula of the delay margin.

**From the formulas to the code.** The model $M\,\dot{\Delta f} = -D\,\Delta f - K\,\Delta f(t - \tau) - \Delta P_{load}$ is `simulate(tau, K)`. The delay margin $\tau^* = \arccos(-a/b) / \sqrt{b^2 - a^2}$ with $a = D/M$ and $b = K/M$ is `delay_margin(a, b)`.

In [ ]:
def simulate(tau, K=K_KW_HZ, step_kw=66.0, T=30.0, dt=0.005):
    """Frequency deviation (Hz) after a load step at t = 1 s, with a controller that sees delay tau."""
    n, lag = int(T / dt), int(round(tau / dt))
    df = np.zeros(n)
    for k in range(1, n):
        seen = df[k - 1 - lag] if k - 1 - lag >= 0 else 0.0
        load = step_kw if k * dt >= 1.0 else 0.0
        df[k] = df[k - 1] + dt * (-D_KW_HZ * df[k - 1] - K * seen - load) / M_KWS_HZ
    return np.arange(n) * dt, df

def delay_margin(a, b):
    """Largest stable delay of x' = -a x - b x(t - tau), valid for b > |a|."""
    return math.acos(-a / b) / math.sqrt(b * b - a * a)

def outcome(t, df, K=K_KW_HZ, step_kw=66.0):
    """'tripped' beyond 2.5 Hz, else 'settled' if the swing around the final value shrinks, else 'growing'."""
    if np.abs(df).max() > 2.5:
        return "tripped"
    final = -step_kw / (D_KW_HZ + K)
    early = np.abs(df[(t >= 10) & (t < 20)] - final).max()
    late = np.abs(df[t >= 20] - final).max()
    return "settled" if late < early else "growing"

a, b = D_KW_HZ / M_KWS_HZ, K_KW_HZ / M_KWS_HZ
tau_star = delay_margin(a, b)
print(f"a = {a:.3f} 1/s, b = {b:.4f} 1/s, delay margin = {tau_star:.3f} s")
print(f"final deviation after 66 kW: {-66 / (D_KW_HZ + K_KW_HZ):.3f} Hz")
for tau in [0.0, 0.4, 0.9, 1.05, 1.1, 1.3]:
    t, df = simulate(tau)
    print(f"tau {tau:.2f} s: lowest {max(df.min(), -9.99):6.3f} Hz -> {outcome(t, df)}")

**Python note.** The element `df[k - 1 - lag]` is the frequency deviation that the controller sees: The value of `lag` steps ago. The list of past values plays the role of the network, which delivers every measurement late. The function `outcome` compares the swing between 10 and 20 s with the swing after 20 s, and an under-frequency relay trips the microgrid beyond 2.5 Hz.

**Exercise D.** Store the delay margin in seconds of a loop with $a = 0.1$ and $b = 1.0$ in `tau_d`.

In [ ]:
tau_d = None   # your answer

check("Exercise D", tau_d)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
tau_d = delay_margin(0.1, 1.0)

check("Exercise D", tau_d)

Part C of the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-08/lab.html) records every load step that you tried with its delay and gain. Paste the copied text into `MY_LOOP`, and the code compares your boundary with the delay margin.

In [ ]:
# Paste the results of lab Part C between the triple quotes.
MY_LOOP = """
"""
if MY_LOOP.strip():
    trials = pd.read_csv(io.StringIO(MY_LOOP.strip()))
    source = "your own trials"
else:
    trials = pd.DataFrame({"trial": range(1, 7), "delay_s": [0.2, 0.8, 1.2, 1.0, 1.1, 1.05],
                           "gain_kw_hz": [125] * 6,
                           "outcome": ["settled", "settled", "tripped", "settled", "growing",
                                       "settled"]})
    source = "a simulated student"
trials["margin_s"] = [round(delay_margin(a, g / M_KWS_HZ), 3) for g in trials["gain_kw_hz"]]
trials["model"] = np.where(trials["delay_s"] < trials["margin_s"], "stable", "unstable")
print("Data:", source)
print(trials.to_string(index=False))
stable = trials[trials["outcome"] == "settled"]["delay_s"]
print("largest delay that settled:", stable.max() if len(stable) else "none",
      "| delay margin of the model:", round(tau_star, 3), "s")

In [ ]:
# Figure: the load step with three delays and the delay margin against the gain
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
for tau, col in [(0.1, "#2E8B57"), (0.8, "#C98510"), (1.15, "#B53F2E")]:
    t, df = simulate(tau, T=25)
    ax[0].plot(t, df, color=col, label=f"tau = {tau} s")
ax[0].set_xlabel("time (s)")
ax[0].set_ylabel("frequency deviation (Hz)")
ax[0].set_ylim(-2.0, 1.0)
ax[0].legend(fontsize=8)
gains = np.linspace(20, 300, 100)
ax[1].plot(gains, [delay_margin(a, g / M_KWS_HZ) for g in gains], color="#1E6FA8")
ax[1].plot(K_KW_HZ, tau_star, "o", color="#B53F2E")
ax[1].annotate(f"K = 125: {tau_star:.3f} s", (K_KW_HZ, tau_star), xytext=(10, 10),
               textcoords="offset points", fontsize=8)
ax[1].set_xlabel("gain K (kW/Hz)")
ax[1].set_ylabel("delay margin (s)")
plt.tight_layout()
plt.show()

**Python note.** The delay margin falls as the gain rises: A stronger controller reacts more to old information. The formula holds only for $b > a$, here for gains above 12 kW/Hz.

**Observe and reflect.** Which delay did you find as the boundary in Part C, and how close was it to the formula? Would you rather lower the gain or shorten the delay when the network is slow?

## 5. The campus microgrid, stage 8

*Lecture: [The campus microgrid, stage 8: The communication design with a delay budget](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-08/lecture.html#the-campus-microgrid-stage-8)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "delay_margin" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-08/"
               "NB08_networks_communication.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Stage 8 joins the four parts of the communication design of the microgrid and checks each against its requirement. The parts are the Modbus bus of the battery container and the MQTT traffic of the meters. The others are the trip messages of the protection relay and the delay of the frequency loop.

In [ ]:
STAGE8 = {}
# 1. The Modbus bus of the battery container
for baud in (9600, 19200):
    STAGE8[f"poll cycle at {baud} baud (ms)"] = round(cycle_time(BUS, baud), 1)
# 2. MQTT traffic of the twelve meters
STAGE8["meter messages per hour, periodic"] = 12 * steps
STAGE8["meter messages per hour, on change"] = sum(len(on_change(r, 5.0, 12)) for r in loads)
STAGE8["broker packets per hour, periodic, QoS 1"] = broker_packets(12 * steps, 3, 1)
# 3. Trip messages of the relay at the point of common coupling
k = len(in_time(goose_times(), transfer=1.5, deadline=10.0))
STAGE8["trip messages within 10 ms"] = k
STAGE8["probability of a missed trip (loss 0.02)"] = float(f"{p_miss(0.02, k):.2e}")
# 4. The frequency loop through three architectures: sampling, uplink, controller, downlink
ARCH = {"local": (0.0, 0.0, 0.02, 0.0), "campus server": (0.2, 0.15, 0.05, 0.1),
        "cloud": (1.0, 0.35, 0.1, 0.35)}
STAGE8["delay margin (s)"] = round(tau_star, 3)
for name, (Ts, up, proc, down) in ARCH.items():
    tau = Ts / 2 + up + proc + down
    STAGE8[f"loop delay, {name} (s)"] = round(tau, 3)
    STAGE8[f"delay margin kept, {name} (%)"] = round(100 * (1 - tau / tau_star), 1)
for key, value in STAGE8.items():
    print(f"{key:<44} {value}")

**Python note.** The loop delay adds half a sampling period, the hold between two samples, to the network and processing times. A negative share of the margin kept means that the loop is unstable.

In [ ]:
# Figure: the delay budget of the three architectures and their load steps
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
left = np.zeros(3)
names = list(ARCH)
for i, (part, col) in enumerate(zip(["half sample", "uplink", "controller", "downlink"],
                                    ["#9DB7CC", "#1E6FA8", "#C98510", "#2E8B57"])):
    vals = np.array([ARCH[n][0] / 2 if i == 0 else ARCH[n][i] for n in names])
    ax[0].barh(names, vals, left=left, color=col, label=part)
    left += vals
ax[0].axvline(tau_star, color="#B53F2E", ls="--", label=f"delay margin {tau_star:.3f} s")
ax[0].set_xlabel("loop delay (s)")
ax[0].legend(fontsize=7, loc="lower right")
for n, col in zip(names, ["#2E8B57", "#1E6FA8", "#B53F2E"]):
    t, df = simulate(STAGE8[f"loop delay, {n} (s)"], T=20)
    ax[1].plot(t, np.clip(df, -2.5, 1.5), color=col, label=n)
ax[1].set_xlabel("time (s)")
ax[1].set_ylabel("frequency deviation (Hz)")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Exercise E.** Store the share of the delay margin, in percent, that the loop through the campus server keeps, as kept in `STAGE8`, in `kept`.

In [ ]:
kept = None   # your answer

check("Exercise E", kept)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
kept = STAGE8["delay margin kept, campus server (%)"]

check("Exercise E", kept)

**Observe and reflect.** Which part of the delay budget of the campus server would you shorten first, and how? Which function of the microgrid could still run in the cloud?

## 6. Your application: The application switch

*Lecture: [Your application](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-08/lecture.html#your-application)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "STAGE8" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-08/"
               "NB08_networks_communication.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Other cyber-physical systems face the same questions. How long does a fieldbus need to read its devices? How much traffic does a broker carry, and how much delay does a control loop tolerate? The functions of this week answer them after one change of a name.

The systems and their numbers are illustrative. Set `APP` to one of the six names and run the cell.

In [ ]:
APP = "factory_line"
# choose: microgrid, factory_line, water_network, smart_building, rail_crossing, icu_monitoring
APPS = {
    "microgrid": ("battery container", BUS, 19200, 12, 25, (D_KW_HZ / M_KWS_HZ, K_KW_HZ / M_KWS_HZ), 0.40),
    "factory_line": ("conveyor drives", [("drive", 1, 6)] * 8, 115200, 40, 1, (0.5, 4.0), 0.12),
    "water_network": ("pumping station", [("flow meter", 1, 4), ("pump", 2, 12), ("valve", 3, 2)],
                      9600, 30, 60, (0.02, 0.1), 4.0),
    "smart_building": ("air handling unit", [("sensor", 1, 2)] * 10, 19200, 120, 30, (0.002, 0.01), 20.0),
    "rail_crossing": ("crossing controller", [("axle counter", 1, 4), ("barrier", 2, 2)],
                      38400, 4, 1, (1.0, 6.0), 0.08),
    "icu_monitoring": ("bedside monitors", [("monitor", 1, 16)] * 6, 57600, 24, 2, (0.3, 1.5), 0.6)}

name, bus, baud, n_pub, period, (a_app, b_app), delay = APPS[APP]
cycle = cycle_time(bus, baud)
margin = delay_margin(a_app, b_app)
print(f"{APP}: {name}, {len(bus)} devices at {baud} baud -> poll cycle {cycle:.1f} ms")
print(f"  {n_pub} publishers every {period} s -> {n_pub * 3600 // period:,} messages per hour")
print(f"  loop: a = {a_app}, b = {b_app} -> delay margin {margin:.3f} s; loop delay {delay} s "
      f"-> {'stable' if delay < margin else 'UNSTABLE'}, margin kept {100 * (1 - delay / margin):.0f} %")

**Python note.** Each entry holds the fieldbus, the number and period of the publishers, the two constants $a$ and $b$ of a first-order loop and its total delay. The same three functions as in Sections 1, 2 and 4 do the rest.

**Observe and reflect.** Which part of your system is closest to its limit: The fieldbus, the broker or the control loop? What would you change first?

## Application challenges

Each challenge transfers the ideas of the week to a field of computing. Choose the one closest to your interests and solve it in a copy of this Colab notebook.

| Field | Challenge |
|---|---|
| Software engineering | Write a function that parses a Modbus response frame, checks its CRC and returns the register values, and test it with a damaged frame. |
| Embedded systems | Compute the poll cycle of eight sensors on an RS-485 bus at four baud rates and choose the slowest rate that meets a cycle of 100 ms. |
| Internet of Things | Design the topic tree of a smart home with ten devices and write filters for a dashboard, an alarm service and a logger. Check them with `matches`. |
| Computer networks | Simulate a gateway that serves two classes of messages and give the trip messages priority. Compare the 99th percentile of both classes. |
| Control engineering | Add a sampling period to `simulate` by holding each measurement for $T_s$ seconds and find the delay margin again for $T_s$ = 0.2, 0.5 and 1 s. |
| Data science | Simulate a month of meter readings and choose the deadband that keeps the error of a daily energy total below 0.5 percent with the fewest messages. |
| Cybersecurity | List what an attacker on the RS-485 bus could read and write with function codes 03 and 06. Which part of this week would Week 11 have to protect? |
| Artificial intelligence | A model in the cloud predicts the load every minute. Use the delay budget of stage 8 to decide which decisions it may take and which must stay local. |

## After the Colab notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-08/lab.html), tabs *Check yourself* and *Reflect and export*. The weekly task and the research assignment are on the [week overview](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/weeks/week-08).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>